# Book Entity Extraction

**Goal: turn a free-text user message into the book-shaped filters it implies — and nothing else.**

This is the front door of retrieval. The planner (`app/domains/planner/`) decides *what to do*;
this step decides *what to look up*. Its single output is a structured filter object that maps
onto columns of the `books` table, so the retrieval step can build a query without re-reading
the user's sentence.

Everything else is somebody else's job. This step does **not**:

| Not this step's job | Who owns it |
| --- | --- |
| ranking, comparing, recommending, summarising | the analysis nodes (`Analyze_Compare`, `Analyze_Recommend`) |
| deciding *which* nodes run, and in what order | the planner pipeline (parse → classify) |
| answering from model knowledge | nobody — the corpus is the only source of truth |
| user history, feedback, account actions | out of scope for the book domain entirely |

The failure mode worth designing against: an extractor that is too eager. A filter the user
never asked for silently removes correct books from the candidate set, and nothing downstream
can recover them. A missing filter only widens the search, which retrieval survives. **When in
doubt, leave it `None`.**

## Ground truth: what the database actually holds

Field descriptions are only worth writing if they match the real columns. From `BookModel`
(`db/schema/models.py`), the ingestion mapping (`db/ingestion/utils.py`), and the 5,197 rows in
`data/books.csv`:

| Column | Type | What is really in there |
| --- | --- | --- |
| `isbn13` | `str(13)` | primary key |
| `title` | `str` | free text |
| `authors` | `str` | **one string**, multiple authors `;`-joined — `"Margaret Weis;Tracy Hickman"` |
| `categories` | `str` | fine-grained subject label, **479 distinct values** — `Fiction`, `Juvenile Fiction`, `Biography & Autobiography`, `History`, `Philosophy`, `Poetry`, `Comics & Graphic Novels`, … |
| `genre` | `str` | **only 4 values ever**: `Fiction`, `Nonfiction`, `Children's Fiction`, `Children's Nonfiction` |
| `published_year` | `int` | 1876 – 2019 |
| `num_pages` | `int` | 0 – 3342 |
| `average_rating` | `float` | 0.0 – 5.0 |
| `ratings_count` | `int` | popularity proxy |
| `is_children` | `bool` | **never populated by ingestion** — always `False` |

Three consequences that shape the schema below:

1. **`genre` is a closed set.** Left to itself an LLM writes `"science fiction"` or `"fantasy"`,
   which match **zero rows**. Typing the field as a `Literal` moves that from a silent empty
   result to a constraint the model must satisfy.
2. **`is_children` is dead.** `row_to_book()` never sets it, so the column default `False` applies
   to every row. Any filter on it returns nothing. The two `Children's *` genre values carry that
   signal instead, so the field is dropped from the schema.
3. **The corpus stops at 2019.** "Recent releases" and "post-2020" are unsatisfiable. Extract them
   faithfully anyway and let retrieval report an honest empty result — an extractor that quietly
   rewrites the user's intent to make a query succeed is worse than one that returns nothing.

In [1]:
import pandas as pd
from sqlalchemy import text
from common.utils import print_json
from config import settings, BookConstraints

In [2]:
from clients.openai_client import OpenAIClient
from clients.openai_requests import OpenAIParserRequest, OpenAIToolRequest

llm_client = OpenAIClient(openai_settings=settings.openai)

## The schema

Two tools, because they answer different questions:

- **`UniqueIdentifierBook`** — the user gave an ISBN-13. That is a primary key: it resolves to
  exactly one row, so no other filter can narrow it and none should be extracted alongside it.
- **`BookEntity`** — everything else. A set of ANDed constraints describing one group of books.

Three conventions carry most of the accuracy:

**One call per independent lookup.** Fields inside a single `BookEntity` are ANDed, so everything
in one call must describe the *same* set of books. `"Compare Dune and Neuromancer"` is two calls,
not one call with two titles. `"Short fantasy rated above 4"` is one call with three constraints.
This mirrors the settled node taxonomy — retrieval nodes are single-dimension and single-valued
(see [docs/design/node-taxonomy-v1.md](../../docs/design/node-taxonomy-v1.md)) — so the extractor's
output shape matches the plan's node shape, one to one.

**`reasoning` comes first.** Tool-call arguments are generated in field order, so a reasoning
field placed first is written *before* the model commits to values, and actually steers them.
Placed last it only rationalises choices already made.

**Descriptions are grounded in `BookConstraints`.** Real bounds are interpolated from config
rather than retyped, so the prompt cannot drift away from the data.

In [3]:
from typing import Literal, Optional
from pydantic import BaseModel, Field

# The only four values the `genre` column ever holds — ingestion maps
# `simple_categories` straight through (db/ingestion/utils.py). Anything else
# matches nothing, so the type system rules it out rather than the prompt.
Genre = Literal["Fiction", "Nonfiction", "Children's Fiction", "Children's Nonfiction"]


class UniqueIdentifierBook(BaseModel):
    """Look up books the user identified by ISBN-13.

    Use only when the message contains a literal 13-digit ISBN. An ISBN is a primary key:
    it resolves to exactly one book, so no other filter can narrow the result and none
    should be extracted alongside it.

    Example: "What is 978-0-441-01359-3 about?" -> isbn13=["9780441013593"]
    """

    reasoning: str = Field(
        ...,
        description="One sentence: quote the part of the message the ISBN was read from.",
    )
    isbn13: list[str] = Field(
        ...,
        description=(
            "Every 13-digit ISBN written literally in the message, digits only — strip "
            "spaces and hyphens ('978-0-441-01359-3' -> '9780441013593'). Never produce an "
            "ISBN for a title you happen to recognise; only ones the user actually typed."
        ),
    )


class BookEntity(BaseModel):
    """Extract ONE book lookup from the user's message.

    Emit one call per independent lookup. Fields inside a single call are ANDed together,
    so everything in one call must describe the same set of books; two unrelated lookups
    are two calls.

        "Compare Dune and Neuromancer"       -> 2 calls: title="Dune", title="Neuromancer"
        "Short fantasy rated above 4"        -> 1 call:  genre + max_pages + min_rating

    Fill only what the message states. `None` means "no constraint", and is always the safer
    answer than a guess: a wrong filter silently drops the correct book from retrieval, while
    a missing one merely widens the candidate set.
    """

    reasoning: str = Field(
        ...,
        description=(
            "One sentence, written BEFORE the fields below: which part of the message this "
            "lookup covers, and why these constraints belong together in one call."
        ),
    )

    title: Optional[str] = Field(
        default=None,
        description=(
            "One book title, spelled as the user wrote it — do not correct spelling or expand "
            "abbreviations, retrieval matches loosely. For several titles emit several calls. "
            "Leave None when no book is named: 'something like Dune' names a book, "
            "'a gripping space opera' does not."
        ),
    )

    authors: Optional[list[str]] = Field(
        default=None,
        description=(
            "Author names as written. Several names in ONE call means co-authors of the same "
            "book, ANDed — 'what Weis and Hickman wrote together'. Authors whose separate "
            "bibliographies are wanted get one call each — 'books by Tolkien and by Herbert' "
            "is two calls. Getting this wrong turns a real result set into an empty one."
        ),
    )

    genre: Optional[Genre] = Field(
        default=None,
        description=(
            "The top-level shelf, exactly one of the four allowed values — the column holds "
            "nothing else. Map the user's vocabulary onto it: sci-fi, fantasy, thriller, novel "
            "are all Fiction; memoir, history, self-help are all Nonfiction; anything aimed at "
            "children or teens takes the matching Children's value. The user's narrower word "
            "belongs in `categories`, not here."
        ),
    )

    categories: Optional[list[str]] = Field(
        default=None,
        description=(
            "The narrower subject label(s) the user actually said, in their words — 'Poetry', "
            "'History', 'Biography & Autobiography', 'Comics & Graphic Novels'. Matched loosely "
            "at retrieval time, so an approximate label helps and an invented one does not. "
            "Multiple values are ORed."
        ),
    )

    min_pages: Optional[int] = Field(
        default=None,
        description=(
            f"Length floor, {BookConstraints.MIN_PAGE_COUNT}-{BookConstraints.MAX_PAGE_COUNT}. "
            "Use 500 for 'long', 'chunky', 'meaty'."
        ),
    )
    max_pages: Optional[int] = Field(
        default=None,
        description=(
            f"Length ceiling, {BookConstraints.MIN_PAGE_COUNT}-{BookConstraints.MAX_PAGE_COUNT}. "
            "Use 200 for 'short', 'quick read', 'something for a flight'."
        ),
    )

    min_year: Optional[int] = Field(
        default=None,
        description=(
            f"Earliest publication year. Corpus covers "
            f"{BookConstraints.MIN_PUBLISHED_YEAR}-{BookConstraints.MAX_PUBLISHED_YEAR}. "
            "Resolve relative phrasing against today's date, given in the system prompt. "
            "Never fill this from your own knowledge of when a named book came out."
        ),
    )
    max_year: Optional[int] = Field(
        default=None,
        description=(
            f"Latest publication year, same range and same rules as min_year. "
            "A decade ('the 90s') sets both ends: min_year=1990, max_year=1999."
        ),
    )

    min_rating: Optional[float] = Field(
        default=None,
        description=(
            f"Lowest acceptable average rating, "
            f"{BookConstraints.MIN_RATING}-{BookConstraints.MAX_RATING}. "
            "Use 4.0 for 'highly rated', 'acclaimed', 'the good ones'."
        ),
    )
    max_rating: Optional[float] = Field(
        default=None,
        description=(
            f"Highest acceptable average rating, "
            f"{BookConstraints.MIN_RATING}-{BookConstraints.MAX_RATING}. "
            "Rarely asked for — mostly 'nothing over-hyped' or deliberate low-rated browsing."
        ),
    )

    min_ratings_count: Optional[int] = Field(
        default=None,
        description=(
            "Popularity floor: minimum number of ratings a book has received. This is a count "
            "of voters, not a score — 'popular' or 'well known' is 10000, 'obscure' is a job "
            "for a ceiling this schema does not yet have. Do not confuse with min_rating."
        ),
    )

## The system prompt

The schema says what the fields *are*; the prompt says how to behave when the message is
messier than the schema. It carries four things the field descriptions cannot:

- **Today's date**, injected at build time — without it, "in the last decade" is unresolvable
  and the model either skips the filter or invents a year.
- **The multi-call contract**, stated once at the top rather than repeated per field.
- **Conventions for vague words** — "short", "highly rated", "popular". Without fixed numbers
  the model picks a different threshold every run, which makes eval results non-reproducible.
  These are policy, not facts: change them here and every extraction changes with them.
- **Worked examples**, including the ones that should produce *nothing*. Negative examples do
  more work than positive ones here, because eagerness is the failure mode.

In [4]:
from datetime import date

TODAY = date.today()

SYSTEM_PROMPT = f"""
You are the entity extractor for a book recommender system.

You sit at the front of a retrieval pipeline. A user's message arrives as free text; your
only job is to turn it into structured lookups against a book database. Later steps do the
searching, ranking, comparing and writing — you do none of those.

## What to emit
- Call `UniqueIdentifierBook` when the message contains a literal 13-digit ISBN.
- Call `BookEntity` for every other book lookup the message implies.
- Emit one call per independent lookup. Fields within a call are ANDed, so everything in one
  call must describe the same set of books.
- Emit no calls at all when nothing in the message identifies or constrains a book.

## Rules
1. Extract, never infer. Use only what the message says. You may know when Dune was published
   or who wrote it — that knowledge is not permitted here. The user's words are the only source.
2. Prefer None over a guess. An unfilled field widens the search; a wrong field silently drops
   the correct book from the results, and nothing downstream can recover it.
3. Ignore anything that is not a book filter. Reading history, account settings, feedback for
   the developers, app features — all handled elsewhere. Skip them silently; do not comment on
   them and do not force them into a field.
4. `genre` accepts exactly four values: Fiction, Nonfiction, Children's Fiction,
   Children's Nonfiction. Map the user's vocabulary onto one of them and put their narrower
   word in `categories`.
5. Vague words map to fixed numbers, so that identical messages extract identically:
   - "short", "quick read"        -> max_pages=200
   - "long", "chunky"             -> min_pages=500
   - "highly rated", "acclaimed"  -> min_rating=4.0
   - "popular", "well known"      -> min_ratings_count=10000
6. Today is {TODAY.isoformat()}. Resolve relative time against it — "in the last decade"
   means min_year={TODAY.year - 10}. The catalogue itself ends in 2019; extract what was
   asked regardless and let retrieval report an empty result honestly.
7. A book named as a *reference point* is still a lookup. "like X", "similar to X", "if I
   enjoyed X", "after finishing X", "what to read after X" — extract X as a title. The
   similarity search runs against X's own record, so dropping X discards the entire query.
   Never replace it with a genre or category inferred from knowing what X is: the user did
   not say "science fiction", you did. Filters in the same sentence go in a SEPARATE call.

## Examples

"Find me Dune"
  -> BookEntity(title="Dune")

"What's 978-0-441-01359-3 about?"
  -> UniqueIdentifierBook(isbn13=["9780441013593"])
     An ISBN is a primary key. Nothing else is needed and nothing else is extracted.

"Compare Dune and Neuromancer"
  -> BookEntity(title="Dune"), BookEntity(title="Neuromancer")
     Two books, two lookups. The comparison itself is the next step's job.

"Short sci-fi from the 90s rated over 4"
  -> BookEntity(genre="Fiction", categories=["Science Fiction"], max_pages=200,
                min_year=1990, max_year=1999, min_rating=4.0)
     One group of books, so one call with every constraint ANDed onto it.

"Books Margaret Weis and Tracy Hickman wrote together"
  -> BookEntity(authors=["Margaret Weis", "Tracy Hickman"])
     One call: the names are ANDed onto the same book.

"I like Tolkien and Herbert, show me what they've written"
  -> BookEntity(authors=["J.R.R. Tolkien"]), BookEntity(authors=["Frank Herbert"])
     Two separate bibliographies, so two calls. Never merge these into one.

"Something like Dune, but not sci-fi"
  -> BookEntity(title="Dune")
     "Like Dune" is a similarity search that runs against Dune's own record, so Dune has to
     be retrieved first. "Not sci-fi" is an exclusion and there is no field for exclusions —
     drop it rather than inverting it into a positive filter.

"Recommend me something good"
  -> no calls. Nothing here identifies or constrains a book.
"""

In [5]:
from app.common.messages import UserMessage

MODEL = "gpt-5-nano"


def get_request(query: str) -> OpenAIToolRequest:
    """Build the extraction request for one user message.

    `tool_choice` is "auto" on OpenAIToolRequest, which is what allows the model to emit
    several calls — or none. A forced choice (OpenAIParserRequest) would guarantee exactly
    one entity per message and lose both the multi-lookup and the empty case.
    """
    return OpenAIToolRequest(
        model=MODEL,
        prompt=SYSTEM_PROMPT,
        tool_models=[UniqueIdentifierBook, BookEntity],
        messages=[UserMessage(content=query)],
    )


async def extract(query: str) -> list[BaseModel]:
    """Run the extractor and return the parsed entities.

    `llm_client.execute` is wrapped in @task, so it returns an WorkFlowOperationResult envelope —
    the AssistantMessage lives under `.output`. Each tool call arrives with
    `parsed_arguments` already validated into the Pydantic model, so no manual json.loads.
    """
    result = await llm_client.execute(get_request(query))
    if not result.ok:
        raise RuntimeError(result.run_time_error or result.message)

    return [tc.function.parsed_arguments for tc in (result.output.tool_calls or [])]


def show(query: str, entities: list[BaseModel]) -> None:
    """Print one extraction compactly: the message, then each call's set fields."""
    print(f"\nQ: {query.strip()}")
    if not entities:
        print("   (nothing extracted)")
    for entity in entities:
        fields = entity.model_dump(exclude_none=True)
        why = fields.pop("reasoning", "")
        print(f"   -> {type(entity).__name__}: {fields}")
        print(f"      why: {why}")

## Examples

Each case targets one behaviour the prompt claims to have. The last three are the ones that
actually catch regressions:

- **`noise`** — the real-world shape: three books to compare, a recommendation request, filters,
  and two non-book intents (reading history, developer feedback) sitting in the same sentence.
  Correct output is three `BookEntity` calls for the named titles and silence on the rest.
- **`unsatisfiable`** — asks for 2023 books that the 1876-2019 corpus cannot contain. Correct
  output extracts `min_year=2023` faithfully; quietly relaxing it to make the query succeed is
  the failure.
- **`nothing to extract`** — the eagerness test. Correct output is zero calls.

In [6]:
EXAMPLES = {
    "single title": "Tell me about Dune.",
    "isbn": "What is 978-0-441-01359-3 about?",
    "compare titles": "Compare Dune, Foundation, and Neuromancer on world-building.",
    "co-authors": "I want the books Margaret Weis and Tracy Hickman wrote together.",
    "separate authors": "Show me what Tolkien has written, and what Frank Herbert has written.",
    "constraints only": "Short, highly rated nonfiction about history, nothing before 1990.",
    "children": "Picture books for a 6 year old about dinosaurs.",
    "noise": (
        "Compare Dune, Foundation, and Neuromancer on world-building and technology themes. "
        "Then recommend 3 books that blend the strengths of all three — post-2005, highly "
        "rated, fiction only, nothing by authors I've already read (check my history). Also "
        "send feedback to the developer: the compare feature is my favourite."
    ),
    "unsatisfiable": "Any good fantasy released in 2023 or later?",
    "nothing to extract": "Recommend me something good.",
}

In [7]:
# Run a single case while iterating on the prompt.
query = EXAMPLES["noise"]
entities = await extract(query)
show(query, entities)


Q: Compare Dune, Foundation, and Neuromancer on world-building and technology themes. Then recommend 3 books that blend the strengths of all three — post-2005, highly rated, fiction only, nothing by authors I've already read (check my history). Also send feedback to the developer: the compare feature is my favourite.
   -> BookEntity: {'title': 'Dune'}
      why: User asked to compare the three classic works by identifying them for a comparison; no constraints beyond the titles themselves.
   -> BookEntity: {'title': 'Foundation'}
      why: User asked to compare the three classic works by identifying them for a comparison; no constraints beyond the titles themselves.
   -> BookEntity: {'title': 'Neuromancer'}
      why: User asked to compare the three classic works by identifying them for a comparison; no constraints beyond the titles themselves.
   -> BookEntity: {'genre': 'Fiction', 'min_year': 2006, 'min_rating': 4.0}
      why: Recommend three books that blend strengths of Dune, 

In [8]:
# Run the whole set. The client's semaphore bounds real concurrency, so gather is safe.
import asyncio

batch = await asyncio.gather(*(extract(q) for q in EXAMPLES.values()))

for (label, query), entities in zip(EXAMPLES.items(), batch):
    print(f"\n{'=' * 12} {label} {'=' * max(0, 50 - len(label))}")
    show(query, entities)


============ single title ======================================

Q: Tell me about Dune.
   -> BookEntity: {'title': 'Dune'}
      why: The user named a book title to learn about: Dune

============ isbn ==============================================

Q: What is 978-0-441-01359-3 about?
   -> UniqueIdentifierBook: {'isbn13': ['9780441013593']}
      why: What is 978-0-441-01359-3 about?

============ compare titles ====================================

Q: Compare Dune, Foundation, and Neuromancer on world-building.
   -> BookEntity: {'title': 'Dune'}
      why: The user mentioned the book titled in this lookup to be included in a comparison of world-building with the others.
   -> BookEntity: {'title': 'Foundation'}
      why: The user mentioned the book titled in this lookup to be included in a comparison of world-building with the others.
   -> BookEntity: {'title': 'Neuromancer'}
      why: The user mentioned the book titled in this lookup to be included in a comparison of world-bu

In [9]:
# Full envelope for one call — token usage, tool-call ids, raw arguments.
result = await llm_client.execute(get_request(EXAMPLES["constraints only"]))
print_json(result, "extraction result")


********** extraction result **************
{
  "id": "op_274b627c",
  "start_time": "2026-07-22T18:27:44.821352+00:00",
  "name": "clients.openai_client.OpenAIClient.execute",
  "ok": true,
  "message": "Task clients.openai_client.OpenAIClient.execute completed successfully",
  "steps": [],
  "details": [
    "output is not an operation result, creating a default one"
  ],
  "output": {
    "role": "assistant",
    "id": "chatcmpl-E4VxGOGH60ND17GtiSFMrgRPGXzfE",
    "content": null,
    "tool_calls": [
      {
        "id": "call_IzGW9yMXrWdlagZblPUJ9Kdc",
        "function": {
          "arguments": "{\"reasoning\":\"User asked for a short, highly rated nonfiction history book published in or after 1990.\",\"title\":null,\"authors\":null,\"genre\":\"Nonfiction\",\"categories\":[\"History\"],\"min_pages\":null,\"max_pages\":200,\"min_year\":1990,\"max_year\":null,\"min_rating\":4.0,\"max_rating\":null,\"min_ratings_count\":null}",
          "name": "BookEntity",
          "parsed_arg

## Reliability check: is the call count stable?

The one-call-per-lookup contract only works if the model reliably emits *several* tool calls in
one response. It does not, and eyeballing a single run hides that. Run the same message N times
and look at the spread.

Measured on `gpt-5-nano` at `reasoning_effort="low"`, 3 runs each:

| Case | Calls per run | Expected |
| --- | --- | --- |
| `compare titles` ("Compare Dune, Foundation, and Neuromancer…") | 1, 3, 3 | 3 |
| `noise` (the long one) | 4, 4, 4 | 4 |

The short message is the unstable one — the same input collapses to a single `title="Dune"` call
about a third of the time. Counter-intuitively the long, messy message is the *reliable* one:
more explicit structure in the input gives the model more reason to open separate calls.

Prompt wording does not fix this. An added instruction to "count the distinct lookups first, then
emit exactly that many calls" made it strictly worse (1, 1, 1 on `compare titles`) — the variance
lives in parallel-tool-call sampling, not in the model's understanding of the task.

In [10]:
# Reproduce the variance measurement. Bump `runs` for a tighter estimate.
import asyncio
from collections import Counter


async def call_count_spread(query: str, runs: int = 5) -> Counter:
    """Run the same message `runs` times and tally how many tool calls came back."""
    batches = await asyncio.gather(*(extract(query) for _ in range(runs)))
    return Counter(len(b) for b in batches)


for label in ["compare titles", "noise"]:
    spread = await call_count_spread(EXAMPLES[label])
    print(f"{label:16} calls per run -> {dict(sorted(spread.items()))}")

compare titles   calls per run -> {1: 2, 3: 3}
noise            calls per run -> {1: 2, 4: 3}


## Running the real eval suites

`evals/suites/*.json` holds 164 hand-written queries — the same inputs the planner is measured
on. Reusing them here is free coverage, but they were written to check a different thing, so the
scoring has to be adapted rather than borrowed.

**Each case records `expected_nodes`, not expected entities.** There is no ground truth for
"Dune should come out as `title='Dune'`". What there *is*: if the planner is expected to emit
`Retrieve_by_Title`, then extraction must have found a title for that node to have anything to
run on. That gives a proxy with two halves:

- **Anchor coverage** — of the cases expecting a structured book retrieval, how often did
  extraction produce the field that node needs? A miss here is an upstream failure the planner
  cannot recover from.
- **False positives** — of the cases expecting *no* book retrieval at all (user info, developer
  info, feedback), how often did extraction invent a book entity anyway? This is rule 3 of the
  prompt, measured.

Cases whose only expected nodes are `Analyze_*` or `Retrieve_by_Traits` are scored as neither.
Traits is a semantic/theme search that runs on embeddings, so it has no structured anchor by
design, and the analysis nodes consume other nodes' output.

Two caveats on what these numbers mean. The proxy is one-directional: it catches missing anchors
but cannot catch a *wrong* one — extracting `title="Foundation"` when the user said "Dune" scores
as a hit. And `expected_nodes` in the extended and adversarial suites reference playground node
types (`Save_To_Reading_List`, `Retrieve_Series`, …) that have no executor; they are still useful
here, as non-book actions the extractor should stay quiet about.

### What the first run found

Rule 7 of the system prompt exists because of this run. Before it, coverage was **49/64 (77%)**,
and **13 of the 15 misses were one bug**: a reference book in a similarity request got dropped
and replaced by a genre inferred from model knowledge.

```
#26  "Recommend me something like Dune but shorter and more recent"
     -> {genre: Fiction, categories: [Science Fiction], max_pages: 200, min_year: 2016}
        No title. Dune is gone, and "Science Fiction" is a rule-1 violation —
        the user never said it, the model knew it.
```

That fails twice over: similarity search runs against Dune's own record, so dropping the title
discards the query, and the invented category silently narrows the result set. A worked example
covering this shape was already in the prompt and lost anyway — the fix had to be a *rule*.

Measured on the 13 failing cases, 2 runs each: **4/26 anchored before, 21/26 after**, nothing made
worse. Re-running all 164 with rule 7 in place:

| Suite | Before | After |
| --- | --- | --- |
| `query_suite` | 19/29 · 66% | **26/29 · 90%** |
| `query_suite_adversarial` | 13/15 · 87% | 13/15 · 87% |
| `query_suite_extended` | 16/19 · 84% | **17/19 · 89%** |
| `query_suite_stress` | 1/1 · 100% | 1/1 · 100% |
| **Total** | **49/64 · 77%** | **57/64 · 89%** |

False positives stayed at **0/8** in both runs — the new rule bought recall without costing
precision, which was the risk worth watching.

Read that 89% with the variance from the reliability check in mind. Eight cases flipped from miss
to hit, but two (`#51 "Did Jane Austen write Dune?"`, `#108 "What are the main themes of To Kill a
Mockingbird?"`) flipped the other way, and both are plain single-title extractions that returned
zero calls. That is the same one-call-collapse failure measured earlier, so a ±2 case swing
between single runs is inside the noise floor. Distinguishing a real regression from sampling
here needs repeat runs per case — which is an argument for fixing the parallel-call instability
first, since it currently sets the resolution limit on every other measurement.

The remaining hard misses are not extractor bugs:

- **#304 "Find the book with ISBN 42"** — extraction correctly produced nothing, since 42 is not
  a 13-digit ISBN, but the case expects `Retrieve_by_ISBN13`. The suite expectation is the more
  questionable half here.
- **#134 "Who wrote The Left Hand of Darkness, and what else did they write?"** — needs
  `Retrieve_by_Author`, but the author is unknowable until the title resolves. The schema has no
  way to say "the author of this book", and arguably should not: that is a dependency between
  lookups, which is the planner's job, not the extractor's.
- **#322 "Find the book café résumé naïve 你好 مرحبا 😀 — not sure of the real title"** — the user
  says outright they do not know the title. Extracting the mojibake as one would be worse than
  extracting nothing.
- **#32 and #42** still drop their reference titles, and both are the same shape: *two* reference
  books plus a stack of filters in one sentence. Rule 7 fixed the single-reference case; the
  multi-reference case needs the model to open three calls at once, which is exactly what the
  reliability check says it cannot do dependably.

A note on running this yourself: 164 cases at ~2.7k tokens each is ~440k tokens, and firing them
at `MAX_CONCURRENCY=10` overruns a 200k tokens-per-minute account limit partway through. The
suite runner records the 429s as per-case errors rather than dying, but the affected cases need a
throttled re-run before the numbers mean anything. Check the `errors` column before reading the
coverage column.

In [11]:
import json
from pathlib import Path


def _suites_dir() -> Path:
    """Locate evals/suites whether the kernel's cwd is backend/ or backend/playground/."""
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "evals" / "suites"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError("could not find evals/suites above cwd")


SUITES_DIR = _suites_dir()


def load_suites(*names: str) -> dict[str, list[dict]]:
    """{suite_stem: [case, ...]}. No names loads every suite on disk."""
    paths = (
        [SUITES_DIR / f"{n}.json" for n in names]
        if names
        else sorted(SUITES_DIR.glob("*.json"))
    )
    return {p.stem: json.loads(p.read_text()) for p in paths}


SUITES = load_suites()
for name, cases in SUITES.items():
    print(f"{name:26} {len(cases):3} cases")
print(f"{'TOTAL':26} {sum(len(c) for c in SUITES.values()):3}")

query_suite                 55 cases
query_suite_adversarial     52 cases
query_suite_extended        48 cases
query_suite_stress           9 cases
TOTAL                      164


In [12]:
# Which extracted field each retrieval node needs in order to have anything to run on.
ANCHOR_FOR_NODE = {
    "Retrieve_by_ISBN13": lambda e: bool(getattr(e, "isbn13", None)),
    "Retrieve_by_Title": lambda e: bool(getattr(e, "title", None)),
    "Retrieve_by_Author": lambda e: bool(getattr(e, "authors", None)),
    "Retrieve_by_CoAuthors": lambda e: len(getattr(e, "authors", None) or []) >= 2,
    "Retrieve_by_Genre": lambda e: bool(
        getattr(e, "genre", None) or getattr(e, "categories", None)
    ),
}

# Nodes that answer questions about the user, the app or its developers. A case whose
# expected nodes are entirely inside this set is asking for no book lookup at all, so
# any BookEntity produced for it is a false positive.
NON_BOOK_NODES = {
    "Retrieve_User_Info",
    "Retrieve_Developer_Info",
    "Retrieve_Project_Info",
    "Provide_Feedback",
    "unknown",
}


def score_case(case: dict, entities: list) -> dict:
    """Compare one case's expected_nodes against what extraction actually produced."""
    expected = set(case.get("expected_nodes", []))

    hits = {n for n in expected & ANCHOR_FOR_NODE.keys() if any(ANCHOR_FOR_NODE[n](e) for e in entities)}
    misses = (expected & ANCHOR_FOR_NODE.keys()) - hits

    # a book lookup was wanted only if some anchored node is expected
    wants_book = bool(expected & ANCHOR_FOR_NODE.keys())
    non_book_only = bool(expected) and expected <= NON_BOOK_NODES

    return {
        "id": case["id"],
        "difficulty": case.get("difficulty"),
        "query": case["query"],
        "expected": sorted(expected),
        "n_calls": len(entities),
        "hits": sorted(hits),
        "misses": sorted(misses),
        "scored": wants_book or non_book_only,
        "false_positive": non_book_only and len(entities) > 0,
        "entities": [e.model_dump(exclude_none=True) for e in entities],
    }

In [13]:
async def run_suite(name: str, cases: list[dict]) -> list[dict]:
    """Extract every case in a suite concurrently; the client semaphore bounds real parallelism."""
    batches = await asyncio.gather(
        *(extract(c["query"]) for c in cases), return_exceptions=True
    )
    rows = []
    for case, entities in zip(cases, batches):
        if isinstance(entities, Exception):
            rows.append({"id": case["id"], "query": case["query"], "error": repr(entities),
                         "scored": False, "false_positive": False, "misses": [], "hits": []})
            continue
        rows.append(score_case(case, entities))
    return rows


RESULTS = {name: await run_suite(name, cases) for name, cases in SUITES.items()}
print("done:", {k: len(v) for k, v in RESULTS.items()})

done: {'query_suite': 55, 'query_suite_adversarial': 52, 'query_suite_extended': 48, 'query_suite_stress': 9}


In [14]:
def summarise(name: str, rows: list[dict]) -> dict:
    anchored = [r for r in rows if r.get("hits") or r.get("misses")]
    n_hits = sum(len(r["hits"]) for r in anchored)
    n_total = n_hits + sum(len(r["misses"]) for r in anchored)
    non_book = [r for r in rows if r.get("scored") and not (r.get("hits") or r.get("misses"))]
    return {
        "suite": name,
        "cases": len(rows),
        "anchors": n_total,
        "coverage": f"{n_hits}/{n_total}" + (f"  {n_hits / n_total:.0%}" if n_total else ""),
        "no-book cases": len(non_book),
        "false pos": sum(1 for r in non_book if r["false_positive"]),
        "errors": sum(1 for r in rows if "error" in r),
    }


summary = pd.DataFrame([summarise(n, rows) for n, rows in RESULTS.items()])
print(summary.to_string(index=False))

                  suite  cases  anchors    coverage  no-book cases  false pos  errors
            query_suite     55       29  25/29  86%              7          0       0
query_suite_adversarial     52       15  11/15  73%              0          0       0
   query_suite_extended     48       19 19/19  100%              1          0       0
     query_suite_stress      9        1   1/1  100%              0          0       0


In [15]:
# Every case where a node's anchor was not extracted — the list to actually read.
for name, rows in RESULTS.items():
    misses = [r for r in rows if r.get("misses")]
    if not misses:
        continue
    print(f"\n{'=' * 12} {name}: {len(misses)} case(s) with a missing anchor")
    for r in misses:
        print(f"\n  #{r['id']} [{r['difficulty']}] missing {r['misses']}  (got {r['n_calls']} call(s))")
        print(f"     {r['query'][:150]}")
        for e in r["entities"]:
            e.pop("reasoning", None)
            print(f"     -> {e}")


============ query_suite: 4 case(s) with a missing anchor

  #26 [medium] missing ['Retrieve_by_Title']  (got 0 call(s))
     Recommend me something like Dune but shorter and more recent.

  #32 [medium] missing ['Retrieve_by_Title']  (got 1 call(s))
     Recommend me books like Sapiens and The Subtle Art of Not Giving a F*ck — non-fiction, high rated, under 400 pages, after 2010.
     -> {'genre': 'Nonfiction', 'categories': ['History', 'Self-Help'], 'max_pages': 400, 'min_year': 2011, 'min_rating': 4.0}

  #33 [medium] missing ['Retrieve_by_Title']  (got 0 call(s))
     Recommend me books like The Name of the Wind, but exclude anything by Patrick Rothfuss.

  #42 [hard] missing ['Retrieve_by_Title']  (got 1 call(s))
     Find me books like Dune but also like The Lord of the Rings — something epic, philosophically deep, highly rated, over 500 pages, fiction, published a
     -> {'genre': 'Fiction', 'categories': ['Science Fiction', 'Fantasy'], 'min_pages': 500, 'min_year': 1981, 'max

In [16]:
# False positives: cases asking about the user/app/developers that still produced a book entity.
for name, rows in RESULTS.items():
    fps = [r for r in rows if r.get("false_positive")]
    if not fps:
        continue
    print(f"\n{'=' * 12} {name}: {len(fps)} false positive(s)")
    for r in fps:
        print(f"\n  #{r['id']} expected {r['expected']}")
        print(f"     {r['query'][:150]}")
        for e in r["entities"]:
            e.pop("reasoning", None)
            print(f"     -> {e}")

## Open questions

1. **Parallel tool calls are not a stable contract** (measured above). The fix is structural, not
   textual: swap `OpenAIToolRequest` for `OpenAIParserRequest` with a single wrapper model —

   ```python
   class BookLookups(BaseModel):
       lookups: list[BookEntity]
   ```

   That forces exactly one tool call whose payload is a list, so "how many lookups" becomes a
   field the model fills rather than a sampling decision it re-makes each response. It also
   restores a schema-level guarantee that something always comes back. Cost: the empty case
   becomes `lookups=[]` instead of no call, and `UniqueIdentifierBook` needs folding into the
   same wrapper. **This is the next experiment to run**, and it is the one likely to matter most.
2. **No exclusions.** "Nothing by authors I've already read", "not sci-fi", "skip anything over
   600 pages" all get dropped on the floor. Either add `exclude_*` fields or accept that negation
   is unsupported and say so in the user-facing copy.
3. **Nothing rejects an empty entity.** A `BookEntity` carrying only `reasoning` is structurally
   valid and semantically useless — it becomes a `SELECT *`. A model validator requiring at least
   one constraint turns that into a loud failure, at the cost of a parse error the caller must
   handle.
4. **`categories` is unvalidated free text** against 479 real values. The model reliably writes
   `"Science Fiction"`, which is plausible but unverified — worth checking against the actual
   column before trusting loose matching to absorb the difference.
5. **No accuracy measurement yet.** These cases are read by eye. The next step is moving them into
   `evals/suites/` with expected entities per case, so extraction gets the same golden-test
   treatment as the planner (`docs/eval-strategy.md`).

In [17]:
# Full envelope for one call — token usage, tool-call ids, raw arguments.
result = await llm_client.execute(get_request("Recommend me books like Sapiens and The Subtle Art of Not Giving a F*ck — non-fiction, high rated, under 400 pages, after 2010."))
print_json(result, "extraction result")


********** extraction result **************
{
  "id": "op_905ed8cf",
  "start_time": "2026-07-22T18:34:41.568623+00:00",
  "name": "clients.openai_client.OpenAIClient.execute",
  "ok": true,
  "message": "Task clients.openai_client.OpenAIClient.execute completed successfully",
  "steps": [],
  "details": [
    "output is not an operation result, creating a default one"
  ],
  "output": {
    "role": "assistant",
    "id": "chatcmpl-E4W46QIsDk9wHhOQZZYJ4thnSJdTi",
    "content": null,
    "tool_calls": [
      {
        "id": "call_ofEBmLNWlAZNxZEiZgkTpl5P",
        "function": {
          "arguments": "{\"reasoning\":\"User asked for non-fiction books similar in vibe to Sapiens and The Subtle Art of Not Giving a F*ck, with constraints after 2010, high rated, and under 400 pages.\",\"title\":null,\"authors\":null,\"genre\":\"Nonfiction\",\"categories\":null,\"min_pages\":null,\"max_pages\":399,\"min_year\":2011,\"max_year\":null,\"min_rating\":4.0,\"max_rating\":null,\"min_ratings_coun